# Price Sensitivity Analyzer — Model Training

This notebook walks through generating the synthetic dataset, exploring it, training and comparing several regression models to predict `Demand`, and saving the best model bundle for the Streamlit app.

For a scripted, repeatable version of this pipeline, see `train_model.py` in the project root.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from utils.preprocessing import load_dataset, clean_dataset, build_feature_matrix
from utils.metrics import evaluate_model, compare_models

## 1. Load & clean the dataset

In [ ]:
raw_df = load_dataset('../dataset/price_sensitivity_dataset.csv')
df = clean_dataset(raw_df)
df.head()

In [ ]:
df.describe()

## 2. Quick exploratory look at price vs demand

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
sample = df.sample(2000, random_state=42)
plt.scatter(sample['Price'], sample['Demand'], alpha=0.3, s=10)
plt.xlabel('Price')
plt.ylabel('Demand')
plt.title('Price vs Demand (sample)')
plt.show()

## 3. Build features and split data

In [ ]:
X, y, encoders = build_feature_matrix(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

## 4. Train and compare candidate models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.08, random_state=42),
}

try:
    from xgboost import XGBRegressor
    models['XGBoost'] = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.08, random_state=42, n_jobs=-1)
except ImportError:
    print('xgboost not installed - skipping')

results = {}
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    metrics = evaluate_model(y_test, preds)
    results[name] = metrics
    fitted[name] = model
    print(name, metrics)

In [ ]:
best_name = compare_models(results)
best_model = fitted[best_name]
print('Best model:', best_name, results[best_name])

## 5. Feature importance (tree-based models only)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values()
    importances.plot(kind='barh', figsize=(6,4), color='#00D4B4')
    plt.title('Feature Importance')
    plt.show()
else:
    print('Selected model has no feature_importances_ attribute.')

## 6. Save the model bundle for the Streamlit app

In [ ]:
import joblib
from utils.preprocessing import FEATURE_COLUMNS

avg_cost_ratio = (df['Cost_Price'] / df['Price']).groupby(df['Category']).mean().to_dict()

bundle = {
    'model': best_model,
    'encoders': encoders,
    'feature_columns': FEATURE_COLUMNS,
    'metrics': results,
    'best_model_name': best_name,
    'avg_cost_ratio': avg_cost_ratio,
}

joblib.dump(bundle, '../models/price_model.pkl')
print('Saved ../models/price_model.pkl')